![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Redis Cloud Agent Memory with the NVIDIA NeMo Agent Toolkit

## Introduction

This is the managed-cloud twin of `06_nemo_agent_toolkit_redis.ipynb`. Same agent, same [**nemo-agent-toolkit-redis**](https://github.com/redis-developer/nemo-agent-toolkit-redis) wiring — but instead of running the open-source [Agent Memory Server](https://github.com/redis/agent-memory-server) ourselves, we point the [**NVIDIA NeMo Agent Toolkit**](https://github.com/NVIDIA/NeMo-Agent-Toolkit) at **[Redis Cloud Agent Memory](https://redis.io/agent-memory/)**, a fully managed service.

No Docker, no server to operate, no worker to babysit — just an endpoint and an API key.

## How this differs from the self-hosted recipe

Notebook 06 runs the open-source Agent Memory Server and uses its **auto-memory** workflow, which hydrates the prompt from *working memory* on every turn. Redis Cloud Agent Memory exposes a **different API** (a dedicated SDK with an `api_key` + `store_id`), and the toolkit integrates it as a **long-term memory store**.

So this recipe uses the toolkit's **tool-based memory** pattern instead: a NAT `react_agent` that decides when to call two tools — `add_memory` (store a fact) and `get_memory` (recall facts) — both backed by the managed cloud store.

| | Self-hosted (nb 06) | Redis Cloud (this nb) |
|---|---|---|
| Server | you run it via Docker | fully managed |
| Backend `_type` | `redis_agent_memory_backend` | `cloud_redis_agent_memory` |
| Auth | disabled (dev) | `api_key` + `store_id` |
| Memory model | automatic working-memory hydration | explicit `add_memory` / `get_memory` tools |
| Workflow | `redis_agent_memory_auto_memory` | `react_agent` |

The behavior a user sees is the same — the agent remembers facts across turns — but memory is managed through explicit tool calls rather than automatic hydration.

## Let's Begin
<a href="https://colab.research.google.com/github/redis-developer/redis-ai-resources/blob/main/python-recipes/agents/07_nemo_agent_toolkit_redis_cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Prerequisites

1. [Create a database on Redis Cloud](https://redis.io/docs/latest/operate/rc/databases/create-database) (a [free account](https://redis.io/try-free/) works).
2. [Create an Agent Memory service](https://redis.io/docs/latest/operate/rc/context-engine/agent-memory/create-service) for that database.
3. From the service's **Configuration** page, grab the **API endpoint**, the **Store ID**, and the **API key** (shown only once at creation — [regenerate](https://redis.io/docs/latest/operate/rc/context-engine/agent-memory/view-service#replace-service-api-key) if lost).
4. An **OpenAI API key** for the chat LLM.

In [6]:
# NBVAL_SKIP
# Cloud backend requires nemo-agent-toolkit-redis >= 0.3.0 (adds cloud_redis_agent_memory).
# nvidia-nat-langchain provides the react_agent used for tool-based memory.
%pip install -q "nemo-agent-toolkit-redis>=0.3.0" nvidia-nat-langchain requests

Note: you may need to restart the kernel to use updated packages.


## Set environment variables

Point the toolkit at your managed endpoint and supply the API key.

In [7]:
# NBVAL_SKIP
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# From your Agent Memory service Configuration page on Redis Cloud:
#   - API endpoint (base URL) and Store ID from the service settings
#   - API key is shown only once, when you create/replace the service key
if not os.environ.get("AGENT_MEMORY_ENDPOINT"):
    os.environ["AGENT_MEMORY_ENDPOINT"] = input("Agent Memory API endpoint (e.g. https://<region>.memory.redis.io): ").strip()
if not os.environ.get("AGENT_MEMORY_STORE_ID"):
    os.environ["AGENT_MEMORY_STORE_ID"] = input("Agent Memory Store ID: ").strip()
if not os.environ.get("AGENT_MEMORY_API_KEY"):
    os.environ["AGENT_MEMORY_API_KEY"] = getpass.getpass("Agent Memory API key: ")

os.environ["NAT_OPENAI_MODEL"] = "gpt-4o-mini"

# One user identity for the whole demo, so stored facts are retrievable across turns.
USER_ID = "demo-user"
SESSION_ID = "demo-session"

## Connect to the managed service

We'll use the cloud SDK directly (via the toolkit's `CloudRedisAgentMemoryEditor`) both to confirm connectivity now and to inspect/clean up memories later. A search that returns without error means the endpoint, store ID, and API key are all good.

In [8]:
# NBVAL_SKIP
from contextlib import asynccontextmanager

from redis_agent_memory import AgentMemory
from nvidia_nat_redis.cloud_redis_agent_memory import CloudRedisAgentMemoryEditor


@asynccontextmanager
async def cloud_memory_editor():
    """Open a CloudRedisAgentMemoryEditor against the managed service."""
    client = AgentMemory(
        os.environ["AGENT_MEMORY_ENDPOINT"],
        store_id=os.environ["AGENT_MEMORY_STORE_ID"],
        api_key=os.environ["AGENT_MEMORY_API_KEY"],
    )
    try:
        yield CloudRedisAgentMemoryEditor(client)
    finally:
        await client.__aexit__(None, None, None)


async with cloud_memory_editor() as editor:
    await editor.search(query="connectivity check", user_id=USER_ID, top_k=1)
print("Connected to Redis Cloud Agent Memory ✓")

Package metadata not found for nvidia-nat


Connected to Redis Cloud Agent Memory ✓


## Define the NAT workflow

The config wires three things together:

- **`memory.redis_memory`** — the `cloud_redis_agent_memory` backend, pointed at your managed endpoint with `api_key` + `store_id`.
- **`get_memory` / `add_memory`** — the two memory tools, both bound to that backend.
- **`react_agent`** — the agent that calls those tools. We enable `use_native_tool_calling` (OpenAI function calling, more reliable than ReAct text parsing) and bake the demo `user_id` into `additional_instructions` so the agent stores and recalls under one consistent identity.

Environment variables (`${...}`) are resolved from the values set above.

In [9]:
config_yaml = """general:
  telemetry:
    enabled: false

llms:
  openai_llm:
    _type: openai
    model_name: ${NAT_OPENAI_MODEL:-gpt-4o-mini}
    temperature: 0.0

memory:
  redis_memory:
    _type: cloud_redis_agent_memory
    base_url: ${AGENT_MEMORY_ENDPOINT}
    api_key: ${AGENT_MEMORY_API_KEY}
    store_id: ${AGENT_MEMORY_STORE_ID}

functions:
  get_memory:
    _type: get_memory
    memory: redis_memory
    description: "Retrieve stored facts about the user. Call before answering anything personal."
  add_memory:
    _type: add_memory
    memory: redis_memory
    description: "Store a durable fact or preference the user shares."

workflow:
  _type: react_agent
  tool_names: [get_memory, add_memory]
  description: "A chat agent using Redis Cloud Agent Memory for long-term memory."
  llm_name: openai_llm
  use_native_tool_calling: true
  additional_instructions: >-
    You assist the user whose user_id is "demo-user". ALWAYS pass
    user_id="demo-user" to get_memory and add_memory. Never ask the user for an
    id. When the user shares any durable preference or fact, immediately call
    add_memory with that fact. Before answering any question about the user's
    preferences or history, first call get_memory to retrieve relevant facts.
"""

with open("nat_config.yml", "w") as f:
    f.write(config_yaml)
print("wrote nat_config.yml")

wrote nat_config.yml


## Run the agent

`run_workflow` runs a single turn. We pass `session_kwargs` so NAT sets the runtime `user_id` / `conversation_id`; the memory tools use that `user_id` to scope what they store and recall in the cloud.

In [10]:
# NBVAL_SKIP
import asyncio
from pathlib import Path

from nat.utils import run_workflow

CONFIG_FILE = Path("nat_config.yml").resolve()


async def chat(prompt: str, user_id: str = USER_ID, conversation_id: str = SESSION_ID) -> str:
    """Run one turn through the react_agent.

    The agent decides when to call add_memory / get_memory, both backed by the
    managed Redis Cloud store and scoped to this user_id.
    """
    result = await run_workflow(
        config_file=CONFIG_FILE,
        prompt=prompt,
        to_type=str,
        session_kwargs={"conversation_id": conversation_id, "user_id": user_id},
    )
    print(f"User: {prompt}")
    print(f"Assistant: {result}\n")
    return result

In [11]:
# NBVAL_SKIP
# A multi-turn conversation. The third turn relies on facts stored in turns 1-2.
await chat("Hi! My name is Justin and my favorite city is Lisbon.")
await chat("I'm a vegetarian, by the way.")
# New turn -> the agent calls get_memory and recalls the earlier facts from Redis Cloud
await chat("Where should I plan a food trip, and what should I keep in mind?")

User: Hi! My name is Justin and my favorite city is Lisbon.
Assistant: Hi Justin! It's great to meet you. I see that your favorite city is Lisbon. What do you love most about it?

User: I'm a vegetarian, by the way.
Assistant: The user is a vegetarian.

User: Where should I plan a food trip, and what should I keep in mind?
Assistant: When planning a food trip, consider destinations known for their vegetarian cuisine. Since your favorite city is Lisbon, you might explore local vegetarian restaurants and markets there. Keep in mind to research the best vegetarian-friendly spots, check for seasonal ingredients, and perhaps look for food festivals that celebrate plant-based dishes. Additionally, consider the local culture and how it influences vegetarian options, as well as any dietary restrictions you may have.



'When planning a food trip, consider destinations known for their vegetarian cuisine. Since your favorite city is Lisbon, you might explore local vegetarian restaurants and markets there. Keep in mind to research the best vegetarian-friendly spots, check for seasonal ingredients, and perhaps look for food festivals that celebrate plant-based dishes. Additionally, consider the local culture and how it influences vegetarian options, as well as any dietary restrictions you may have.'

The third answer reflects the favorite city and diet from earlier turns — recalled from the managed Cloud store via `get_memory`, not from anything passed back in.

## Inspect long-term memory

Query the store directly through the same cloud editor to see what the agent persisted.

In [12]:
# NBVAL_SKIP
# Inspect what the agent stored, straight from the managed store.
async with cloud_memory_editor() as editor:
    memories = await editor.search(query="favorite city and diet", user_id=USER_ID, top_k=5)

for m in memories:
    print(f"- {m.memory}")
if not memories:
    print("(no memories found for this user yet)")

- User's name is Justin and favorite city is Lisbon.
- User's name is Justin and favorite city is Lisbon.
- User is a vegetarian.
- User is a vegetarian.


## Cleanup

Nothing to tear down locally. We delete the demo user's memories to keep the store tidy. To stop incurring cost entirely, delete or pause the Agent Memory service (and its database) from the Redis Cloud console when you're done.

## Summary

We gave a NeMo Agent Toolkit agent persistent memory backed by managed [Redis Cloud Agent Memory](https://redis.io/agent-memory/) — no server to run. Because the cloud service exposes a long-term memory store, we used the toolkit's **tool-based** pattern (`react_agent` + `get_memory` / `add_memory` via the `cloud_redis_agent_memory` backend) rather than notebook 06's automatic working-memory hydration. Same user-facing behavior, a managed backend, and memory managed through explicit tool calls.

## Summary

Same NeMo Agent Toolkit agent, same memory behavior — now backed by managed [Redis Cloud Agent Memory](https://redis.io/agent-memory/). Moving from the self-hosted server to production was a change of `base_url` and an API key, nothing more.